# Quick Smoke Test: Hypothesis-Testing Pipeline

**Purpose:** Runs a rapid 5-second end-to-end verification of the hypothesis-testing pipeline using a self-generated test dataset.

Click **Run All** — it generates a miniature connectome in memory/temp and tests null-graph rewiring, perturbation, and export in ~2 seconds.


In [ ]:
# Cell 1: Environment Setup & sys.path Discovery
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

# Locate codebase root automatically
CODEBASE_ROOT = None
possible_code_paths = [
    Path('/kaggle/input/datasets/jeet7771/flywire-hypothesis-kaggle-package'),
    Path('/kaggle/input/datasets/jeet7771/flywire-codebase'),
    Path('/kaggle/input/flywire-codebase'),
    Path('/kaggle/working'),
    Path(os.getcwd()).resolve(),
]

for p in possible_code_paths:
    if (p / 'hypothesis_testing').exists() and (p / 'configs').exists():
        CODEBASE_ROOT = p
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f'[OK] Discovered codebase at: {CODEBASE_ROOT}')
        break

if CODEBASE_ROOT is None:
    # Fallback to current working directory
    CODEBASE_ROOT = Path(os.getcwd()).resolve()
    sys.path.insert(0, str(CODEBASE_ROOT))
    print(f'[WARN] Using fallback codebase path: {CODEBASE_ROOT}')

CONFIGS_ROOT = str(CODEBASE_ROOT / 'configs')
print(f'[OK] Configs root : {CONFIGS_ROOT}')

# Locate raw datasets folder
KAGGLE_DATA_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')
if KAGGLE_DATA_PATH.exists():
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
    print(f'[OK] Dataset root : {DATASET_ROOT}')
else:
    DATASET_ROOT = 'research_data/raw'
    print(f'[INFO] Dataset root (local/fallback): {DATASET_ROOT}')

print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')


In [ ]:
# Cell 2: Framework Imports
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from hypothesis_testing.config import HypothesisExperimentConfig, ExecutionMode
from hypothesis_testing.runners.hypothesis_experiment_runner import HypothesisExperimentRunner

print('[OK] Hypothesis-testing framework imported successfully.')


In [ ]:
# Cell 3: Create Self-Contained Demo Dataset in /tmp/demo_dataset
# ============================================================
demo_root = Path('/tmp/demo_dataset')
demo_dir = demo_root / 'TEST_v1'
demo_dir.mkdir(parents=True, exist_ok=True)

# Generate 50 sample neurons
nodes_df = pd.DataFrame({
    'root_id': range(1, 51),
    'super_class': ['neuron'] * 50,
    'top_region': ['AL', 'MB', 'LH', 'CX', 'OL'] * 10,
})
nodes_df.to_csv(demo_dir / 'neurons.csv', index=False)

# Generate 200 sample synaptic edges
import numpy as np
rng = np.random.default_rng(42)
src = rng.integers(1, 51, size=200)
tgt = rng.integers(1, 51, size=200)
mask = src != tgt
edges_df = pd.DataFrame({
    'pre_root_id': src[mask],
    'post_root_id': tgt[mask],
    'syn_count': rng.integers(1, 15, size=len(src[mask])),
})
edges_df.to_csv(demo_dir / 'connections.csv', index=False)

print(f'[OK] Created self-contained demo dataset at {demo_dir} (N={len(nodes_df)}, E={len(edges_df)})')


In [ ]:
# Cell 4: Fast Smoke Test Configuration (~2 seconds runtime)
# ============================================================
OUTPUT_ROOT = Path('/tmp/results') / 'smoke_test'

test_config = HypothesisExperimentConfig(
    dataset_name='TEST',
    dataset_root=str(demo_root),
    configs_root=CONFIGS_ROOT,
    execution_mode='null_only',
    null_model_name='degree_preserving',
    null_graph_seeds=[1],
    random_seeds=[1, 2],
    error_model_names=['missed_synapses', 'split_errors'],
    error_rates=[0.0, 0.05],
    analysis_names=['basic_structure'],
    output_root=str(OUTPUT_ROOT),
)

print(f'[OK] Smoke test config ready: Mode={test_config.execution_mode.value}, Dataset={test_config.dataset_name}')


In [ ]:
# Cell 5: Execute Smoke Test
# ============================================================
runner = HypothesisExperimentRunner()
result = runner.run(test_config)

print(f'Pipeline Status : {result.status}')
print(f'Execution Time  : {result.runtime_seconds:.2f}s')

null_csv = OUTPUT_ROOT / 'TEST' / 'null_observations' / 'replicate_level_effects.csv'
if null_csv.exists():
    df = pd.read_csv(null_csv)
    print(f'\n[SUCCESS] Pipeline verified! Generated {len(df)} replicate records.')
    display(df.head(6))
else:
    print(f'[ERROR] Replicate CSV not found. Errors: {result.errors}')
